# Wikipedia Link Prediction Pipeline

**Complete self-contained notebook for Wikipedia link prediction**

This notebook implements a two-step pipeline:
1. **Step 1 - Link Count Prediction**: MLP model predicts how many links a sentence should have
2. **Step 2 - Link Retrieval**: Qdrant vector search retrieves top-K candidate articles

## Features
- Full train/validation/test split
- Complete MLP training pipeline (can retrain from scratch)
- Comprehensive metrics for both steps
- Caching for expensive operations
- Self-contained (no external dependencies on other project files)

## Metrics
- **Step 1**: MAE, RMSE, Exact Accuracy, Within ±1/2/3 Accuracy
- **Step 2**: Recall@1/5/10/20, MRR, Hit Rate, Precision@K

---
# Part 1: Setup and Configuration
---

In [12]:
# =============================================================================
# IMPORTS
# =============================================================================

import json
import re
import os
import gc
import pickle
import time
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union, Any
from dataclasses import dataclass, field, asdict
from urllib.parse import unquote, quote
from datetime import datetime

import numpy as np

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Progress bars
from tqdm.auto import tqdm

# Sentence embeddings
from sentence_transformers import SentenceTransformer

# Vector database
from qdrant_client import QdrantClient

## Configuration

In [13]:
# =============================================================================
# CONFIGURATION
# =============================================================================


@dataclass
class Config:
    """Global configuration for the pipeline."""

    # Paths
    # We use raw JSON (streamed with ijson) to get actual link targets for Step 2
    # JSONL only has link_count, not the actual link hrefs needed for evaluation
    data_path: str = "../fast_data/llm_data.jsonl"  # For Step 1 only (no link targets)
    raw_data_path: str = (
        "../articles_fr_withLinks.json"  # For Step 2 (has link targets)
    )
    cache_dir: str = "./cache"
    results_dir: str = "./results"

    # Qdrant
    qdrant_host: str = "localhost"
    qdrant_port: int = 6333
    collection_name: str = "wikipedia_fr_chunks"

    # Embedding model
    embedding_model: str = "intfloat/multilingual-e5-large"
    embed_dim: int = 1024

    # Data splits
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # Dataset size - keep small to avoid memory issues
    max_sentences: int = 30000  # Max sentences total
    max_articles: int = 2000  # Articles to stream from raw JSON (uses ijson)
    max_sentences_per_article: int = 30  # Limit per article
    min_sentence_length: int = 20

    # Training
    batch_size: int = 256
    learning_rate: float = 5e-4
    weight_decay: float = 0.02
    epochs: int = 150
    patience: int = 40

    # Evaluation
    k_values: List[int] = field(default_factory=lambda: [1, 5, 10, 20])
    top_k_retrieval: int = 20
    min_similarity: float = 0.5

    # Random seed
    seed: int = 42

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# Global config instance
CONFIG = Config()

# Create directories
os.makedirs(CONFIG.cache_dir, exist_ok=True)
os.makedirs(CONFIG.results_dir, exist_ok=True)

print(f"🔧 Device: {CONFIG.device}")
print(f"📁 Cache dir: {CONFIG.cache_dir}")
print(f"📁 Results dir: {CONFIG.results_dir}")

🔧 Device: cuda
📁 Cache dir: ./cache
📁 Results dir: ./results


In [14]:
# =============================================================================
# SET RANDOM SEEDS
# =============================================================================


def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG.seed)
print(f"🎲 Random seed set to {CONFIG.seed}")

🎲 Random seed set to 42


---
# Part 2: Data Classes and Metrics
---

In [15]:
# =============================================================================
# DATA CLASSES
# =============================================================================


@dataclass
class SentenceData:
    """Data for a single sentence."""

    sentence: str
    article_id: int
    article_title: str
    link_count: int
    links: List[Dict] = field(default_factory=list)
    embedding: Optional[np.ndarray] = None


@dataclass
class Step1Metrics:
    """Metrics for Step 1: Link count prediction."""

    mae: float = 0.0
    mse: float = 0.0
    rmse: float = 0.0
    r_squared: float = 0.0
    correlation: float = 0.0
    exact_accuracy: float = 0.0
    within_1_accuracy: float = 0.0
    within_2_accuracy: float = 0.0
    within_3_accuracy: float = 0.0
    n_samples: int = 0

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class Step2Metrics:
    """Metrics for Step 2: Link retrieval."""

    recall_at_1: float = 0.0
    recall_at_5: float = 0.0
    recall_at_10: float = 0.0
    recall_at_20: float = 0.0
    precision_at_1: float = 0.0
    precision_at_5: float = 0.0
    precision_at_10: float = 0.0
    precision_at_20: float = 0.0
    mrr: float = 0.0
    hit_rate_at_1: float = 0.0
    hit_rate_at_5: float = 0.0
    hit_rate_at_10: float = 0.0
    hit_rate_at_20: float = 0.0
    n_samples: int = 0
    n_ground_truth_links: int = 0

    def to_dict(self) -> Dict:
        return asdict(self)

In [16]:
# =============================================================================
# METRICS CALCULATOR
# =============================================================================


class MetricsCalculator:
    """Calculate metrics for both pipeline steps."""

    @staticmethod
    def calculate_step1_metrics(
        predictions: np.ndarray, ground_truth: np.ndarray
    ) -> Step1Metrics:
        """Calculate link count prediction metrics."""
        predictions = np.array(predictions).flatten()
        ground_truth = np.array(ground_truth).flatten()

        # Basic error metrics
        mae = np.mean(np.abs(predictions - ground_truth))
        mse = np.mean((predictions - ground_truth) ** 2)
        rmse = np.sqrt(mse)

        # R-squared
        ss_res = np.sum((ground_truth - predictions) ** 2)
        ss_tot = np.sum((ground_truth - np.mean(ground_truth)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0

        # Correlation
        if (
            len(predictions) > 1
            and np.std(predictions) > 0
            and np.std(ground_truth) > 0
        ):
            correlation = np.corrcoef(predictions, ground_truth)[0, 1]
        else:
            correlation = 0.0

        # Rounded predictions for accuracy metrics
        pred_rounded = np.clip(np.round(predictions), 0, None).astype(int)
        gt_int = ground_truth.astype(int)

        # Accuracy metrics
        exact_accuracy = np.mean(pred_rounded == gt_int)
        within_1 = np.mean(np.abs(pred_rounded - gt_int) <= 1)
        within_2 = np.mean(np.abs(pred_rounded - gt_int) <= 2)
        within_3 = np.mean(np.abs(pred_rounded - gt_int) <= 3)

        return Step1Metrics(
            mae=float(mae),
            mse=float(mse),
            rmse=float(rmse),
            r_squared=float(r_squared),
            correlation=float(correlation) if not np.isnan(correlation) else 0.0,
            exact_accuracy=float(exact_accuracy),
            within_1_accuracy=float(within_1),
            within_2_accuracy=float(within_2),
            within_3_accuracy=float(within_3),
            n_samples=len(predictions),
        )

    @staticmethod
    def calculate_step2_metrics(
        predictions: List[List[int]],
        ground_truth: List[List[int]],
        k_values: List[int] = [1, 5, 10, 20],
    ) -> Step2Metrics:
        """Calculate retrieval metrics."""
        recall_at_k = {k: {"hits": 0, "total": 0} for k in k_values}
        precision_at_k = {k: {"correct": 0, "predicted": 0} for k in k_values}
        hit_at_k = {k: 0 for k in k_values}
        mrr_sum = 0.0
        mrr_count = 0
        total_gt_links = 0
        n_samples = len(predictions)

        for pred_ids, gt_ids in zip(predictions, ground_truth):
            gt_set = set(gt_ids) if gt_ids else set()
            total_gt_links += len(gt_set)

            if not gt_set:
                continue

            for k in k_values:
                top_k = set(pred_ids[:k]) if pred_ids else set()
                hits = len(gt_set & top_k)

                # Recall
                recall_at_k[k]["hits"] += hits
                recall_at_k[k]["total"] += len(gt_set)

                # Precision
                precision_at_k[k]["correct"] += hits
                precision_at_k[k]["predicted"] += (
                    min(k, len(pred_ids)) if pred_ids else 0
                )

                # Hit rate
                if hits > 0:
                    hit_at_k[k] += 1

            # MRR
            for gt_id in gt_set:
                mrr_count += 1
                if pred_ids and gt_id in pred_ids:
                    rank = pred_ids.index(gt_id) + 1
                    mrr_sum += 1.0 / rank

        def safe_div(num, denom):
            return num / denom if denom > 0 else 0.0

        return Step2Metrics(
            recall_at_1=safe_div(recall_at_k[1]["hits"], recall_at_k[1]["total"]),
            recall_at_5=safe_div(recall_at_k[5]["hits"], recall_at_k[5]["total"]),
            recall_at_10=safe_div(recall_at_k[10]["hits"], recall_at_k[10]["total"]),
            recall_at_20=safe_div(recall_at_k[20]["hits"], recall_at_k[20]["total"]),
            precision_at_1=safe_div(
                precision_at_k[1]["correct"], precision_at_k[1]["predicted"]
            ),
            precision_at_5=safe_div(
                precision_at_k[5]["correct"], precision_at_k[5]["predicted"]
            ),
            precision_at_10=safe_div(
                precision_at_k[10]["correct"], precision_at_k[10]["predicted"]
            ),
            precision_at_20=safe_div(
                precision_at_k[20]["correct"], precision_at_k[20]["predicted"]
            ),
            mrr=safe_div(mrr_sum, mrr_count),
            hit_rate_at_1=safe_div(hit_at_k[1], n_samples),
            hit_rate_at_5=safe_div(hit_at_k[5], n_samples),
            hit_rate_at_10=safe_div(hit_at_k[10], n_samples),
            hit_rate_at_20=safe_div(hit_at_k[20], n_samples),
            n_samples=n_samples,
            n_ground_truth_links=total_gt_links,
        )

    @staticmethod
    def print_step1_metrics(metrics: Step1Metrics, title: str = "STEP 1 METRICS"):
        """Pretty print Step 1 metrics."""
        print(f"\n{'=' * 60}")
        print(f"📊 {title}: Link Count Prediction")
        print("=" * 60)
        print(f"  Samples:            {metrics.n_samples:,}")
        print(f"  MAE:                {metrics.mae:.4f}")
        print(f"  RMSE:               {metrics.rmse:.4f}")
        print(f"  R-squared:          {metrics.r_squared:.4f}")
        print(f"  Correlation:        {metrics.correlation:.4f}")
        print("-" * 60)
        print(f"  Exact accuracy:     {metrics.exact_accuracy * 100:.1f}%")
        print(f"  Within ±1:          {metrics.within_1_accuracy * 100:.1f}%")
        print(f"  Within ±2:          {metrics.within_2_accuracy * 100:.1f}%")
        print(f"  Within ±3:          {metrics.within_3_accuracy * 100:.1f}%")
        print("=" * 60)

    @staticmethod
    def print_step2_metrics(metrics: Step2Metrics, title: str = "STEP 2 METRICS"):
        """Pretty print Step 2 metrics."""
        print(f"\n{'=' * 60}")
        print(f"📊 {title}: Link Retrieval")
        print("=" * 60)
        print(f"  Sentences:              {metrics.n_samples:,}")
        print(f"  Ground truth links:     {metrics.n_ground_truth_links:,}")
        print("-" * 60)
        print(f"  Recall@1:               {metrics.recall_at_1:.4f}")
        print(f"  Recall@5:               {metrics.recall_at_5:.4f}")
        print(f"  Recall@10:              {metrics.recall_at_10:.4f}")
        print(f"  Recall@20:              {metrics.recall_at_20:.4f}")
        print("-" * 60)
        print(f"  Precision@1:            {metrics.precision_at_1:.4f}")
        print(f"  Precision@5:            {metrics.precision_at_5:.4f}")
        print(f"  Precision@10:           {metrics.precision_at_10:.4f}")
        print(f"  Precision@20:           {metrics.precision_at_20:.4f}")
        print("-" * 60)
        print(f"  MRR:                    {metrics.mrr:.4f}")
        print(f"  Hit Rate@1:             {metrics.hit_rate_at_1 * 100:.1f}%")
        print(f"  Hit Rate@5:             {metrics.hit_rate_at_5 * 100:.1f}%")
        print(f"  Hit Rate@10:            {metrics.hit_rate_at_10 * 100:.1f}%")
        print(f"  Hit Rate@20:            {metrics.hit_rate_at_20 * 100:.1f}%")
        print("=" * 60)


# Instantiate calculator
metrics_calc = MetricsCalculator()

---
# Part 3: Text Processing
---

In [17]:
# =============================================================================
# TEXT PROCESSING UTILITIES
# =============================================================================


class TextProcessor:
    """Utilities for processing Wikipedia text."""

    @staticmethod
    def extract_links_and_clean_text(text: str) -> Tuple[str, List[Dict]]:
        """
        Extract href links from text and return clean text with links info.

        Input format: '<a href="Moulins%20%28Allier%29">Moulins</a>'
        Returns: (clean_text, list of link dicts with positions)
        """
        if not text:
            return "", []

        # Pattern to match <a href="...">anchor</a>
        href_pattern = (
            r'(?:&lt;|<)a\s+href="([^"]*)"(?:&gt;|>)(.*?)(?:&lt;|<)/a(?:&gt;|>)'
        )

        links = []
        offset = 0

        for match in re.finditer(href_pattern, text, re.IGNORECASE | re.DOTALL):
            href_raw = match.group(1)
            anchor = match.group(2)

            try:
                href_decoded = unquote(href_raw)
            except:
                href_decoded = href_raw

            original_start = match.start()
            clean_start = original_start - offset

            links.append(
                {
                    "anchor": anchor,
                    "href_raw": href_raw,
                    "href_decoded": href_decoded,
                    "start_idx": clean_start,
                    "end_idx": clean_start + len(anchor),
                }
            )

            tag_length = match.end() - match.start()
            anchor_length = len(anchor)
            offset += tag_length - anchor_length

        # Replace all href tags with just anchor text
        clean_text = re.sub(href_pattern, r"\2", text, flags=re.IGNORECASE | re.DOTALL)

        return clean_text, links

    @staticmethod
    def split_into_sentences(text: str) -> List[str]:
        """Split text into sentences using regex."""
        if not text:
            return []

        # French sentence splitting
        sentence_pattern = r"(?<=[.!?])\s+(?=[A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇŒ])"
        sentences = re.split(sentence_pattern, text)

        # Filter and clean
        result = []
        for s in sentences:
            s = s.strip()
            if len(s) >= CONFIG.min_sentence_length:
                result.append(s)

        return result

    @staticmethod
    def extract_sentences_with_links(
        clean_text: str, extracted_links: List[Dict]
    ) -> List[Dict]:
        """
        Extract sentences and match links by character position.
        Returns list of dicts with sentence, links, and link count.
        """
        if not clean_text or not extracted_links:
            return []

        sorted_links = sorted(extracted_links, key=lambda x: x.get("start_idx", 0))
        sentences = TextProcessor.split_into_sentences(clean_text)

        results = []
        current_pos = 0

        for sentence in sentences:
            # Find sentence position in clean text
            sent_start = clean_text.find(sentence, current_pos)
            if sent_start == -1:
                continue
            sent_end = sent_start + len(sentence)
            current_pos = sent_end

            # Find links within this sentence
            links_in_sent = []
            for link in sorted_links:
                link_start = link.get("start_idx", -1)
                if sent_start <= link_start < sent_end:
                    links_in_sent.append(
                        {
                            "anchor": link.get("anchor", ""),
                            "href_decoded": link.get("href_decoded", ""),
                            "href_raw": link.get("href_raw", ""),
                        }
                    )

            results.append(
                {
                    "sentence": sentence,
                    "links": links_in_sent,
                    "num_links": len(links_in_sent),
                }
            )

        return results


# Instantiate processor
text_processor = TextProcessor()

---
# Part 4: Data Loading and Preprocessing
---

In [18]:
# =============================================================================
# DATA LOADING - JSONL FORMAT (RECOMMENDED)
# =============================================================================


def load_from_jsonl(
    jsonl_path: str, max_sentences: int = 50000, cache_path: Optional[str] = None
) -> List[SentenceData]:
    """
    Load preprocessed sentence data from JSONL file.

    NOTE: JSONL files only have link_count, not actual link targets.
    This is fine for Step 1 (link count prediction) but Step 2 needs
    data loaded from raw JSON with actual links.

    Args:
        jsonl_path: Path to JSONL file
        max_sentences: Maximum sentences to load
        cache_path: Optional cache path

    Returns:
        List of SentenceData objects
    """
    # Check cache first
    if cache_path and Path(cache_path).exists():
        print(f"📂 Loading cached data from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"📚 Loading preprocessed data from {jsonl_path}")
    print(
        f"   ⚠️  Note: JSONL has link_count but no link targets (Step 2 needs raw data)"
    )

    all_sentences = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(
            tqdm(f, desc="Loading sentences", total=max_sentences)
        ):
            if i >= max_sentences:
                break

            try:
                data = json.loads(line.strip())

                # Skip very short sentences
                if len(data.get("sentence", "")) < CONFIG.min_sentence_length:
                    continue

                all_sentences.append(
                    SentenceData(
                        sentence=data["sentence"],
                        article_id=int(data.get("article_id", 0)),
                        article_title=data.get("article_title", "Unknown"),
                        link_count=int(data.get("link_count", 0)),
                        links=[],  # Links not stored in JSONL
                    )
                )
            except (json.JSONDecodeError, KeyError) as e:
                continue

    print(f"✅ Loaded {len(all_sentences):,} sentences")

    # Cache if path provided
    if cache_path:
        print(f"💾 Caching to {cache_path}")
        with open(cache_path, "wb") as f:
            pickle.dump(all_sentences, f)

    return all_sentences


def load_from_raw_json_streaming(
    json_path: str,
    max_articles: int = 2000,
    max_sentences: int = 50000,
    cache_path: Optional[str] = None,
) -> List[SentenceData]:
    """
    Load data from raw JSON with ACTUAL LINKS for Step 2 evaluation.

    Uses streaming to avoid memory issues with the 9GB file.
    This extracts sentences WITH their link targets (href_decoded).

    Args:
        json_path: Path to raw articles JSON
        max_articles: Max articles to process
        max_sentences: Max sentences to extract
        cache_path: Optional cache path

    Returns:
        List of SentenceData with actual links
    """
    # Check cache first
    if cache_path and Path(cache_path).exists():
        print(f"📂 Loading cached data with links from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"📚 Streaming raw JSON to extract sentences WITH links")
    print(f"   Source: {json_path}")
    print(f"   Max articles: {max_articles:,}, Max sentences: {max_sentences:,}")

    # Try to use ijson for streaming
    try:
        import ijson

        USE_IJSON = True
        print("   Using ijson for memory-efficient streaming")
    except ImportError:
        USE_IJSON = False
        print("   ⚠️  ijson not available - install with: pip install ijson")
        print("   Falling back to slower method...")

    all_sentences = []
    articles_processed = 0

    if USE_IJSON:
        with open(json_path, "rb") as f:
            parser = ijson.items(f, "item")

            pbar = tqdm(desc="Processing articles", total=max_articles)

            for article in parser:
                if articles_processed >= max_articles:
                    break
                if len(all_sentences) >= max_sentences:
                    break

                article_id = article.get("id", 0)
                article_title = article.get("title", "Unknown")
                text = article.get("text", "")

                if not text or len(text) < 100:
                    continue

                # Extract links and clean text
                clean_text, links = text_processor.extract_links_and_clean_text(text)

                if not clean_text or not links:
                    continue

                # Extract sentences with their links
                sentence_data = text_processor.extract_sentences_with_links(
                    clean_text, links
                )

                for sent_info in sentence_data[: CONFIG.max_sentences_per_article]:
                    if len(all_sentences) >= max_sentences:
                        break

                    # Only keep sentences that have links
                    if sent_info["num_links"] > 0:
                        all_sentences.append(
                            SentenceData(
                                sentence=sent_info["sentence"],
                                article_id=int(article_id) if article_id else 0,
                                article_title=article_title,
                                link_count=sent_info["num_links"],
                                links=sent_info["links"],  # Contains href_decoded!
                            )
                        )

                articles_processed += 1
                pbar.update(1)

                # Periodic memory cleanup
                if articles_processed % 500 == 0:
                    gc.collect()

            pbar.close()
    else:
        # Fallback without ijson - try line by line
        print("   Attempting line-by-line parsing (may not work for all formats)")
        # This would need the file to be in a specific format
        raise ImportError(
            "ijson is required for streaming large JSON files. "
            "Install with: pip install ijson"
        )

    print(
        f"✅ Extracted {len(all_sentences):,} sentences with links from {articles_processed:,} articles"
    )

    # Show link statistics
    total_links = sum(s.link_count for s in all_sentences)
    print(f"   Total ground truth links: {total_links:,}")
    print(f"   Avg links per sentence: {total_links / len(all_sentences):.2f}")

    # Cache the data
    if cache_path:
        print(f"💾 Caching to {cache_path}")
        with open(cache_path, "wb") as f:
            pickle.dump(all_sentences, f)
        print(f"   Cache saved! Next run will be instant.")

    gc.collect()
    return all_sentences

In [21]:
# =============================================================================
# DATA LOADING - RAW JSON (USE ONLY IF REPROCESSING NEEDED)
# =============================================================================


def load_from_raw_json_streaming(
    json_path: str,
    max_articles: int = 2000,
    max_sentences: int = 50000,
    cache_path: Optional[str] = None,
) -> List[SentenceData]:
    if cache_path and Path(cache_path).exists():
        print(f"📂 Loading cached data with links from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"📚 Reading JSONL file: {json_path}")
    all_sentences = []
    articles_processed = 0

    with open(json_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, total=max_articles):
            if articles_processed >= max_articles or len(all_sentences) >= max_sentences:
                break

            try:
                article = json.loads(line.strip())
            except json.JSONDecodeError:
                continue

            article_id = article.get("id", 0)
            article_title = article.get("title", "Unknown")
            text = article.get("text", "")

            if not text or len(text) < 100:
                continue

            clean_text, links = text_processor.extract_links_and_clean_text(text)
            if not clean_text or not links:
                continue

            sentence_data = text_processor.extract_sentences_with_links(clean_text, links)

            for sent_info in sentence_data[:CONFIG.max_sentences_per_article]:
                if len(all_sentences) >= max_sentences:
                    break
                if sent_info["num_links"] > 0:
                    all_sentences.append(
                        SentenceData(
                            sentence=sent_info["sentence"],
                            article_id=int(article_id),
                            article_title=article_title,
                            link_count=sent_info["num_links"],
                            links=sent_info["links"],
                        )
                    )

            articles_processed += 1
            if articles_processed % 500 == 0:
                gc.collect()

    print(f"✅ Extracted {len(all_sentences):,} sentences with links")

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(all_sentences, f)

    return all_sentences


# Alias for backward compatibility
load_wikipedia_data = load_wikipedia_data_streaming

In [22]:
# =============================================================================
# LOAD DATA
# =============================================================================

# For Step 2 evaluation, we NEED the actual links (not just link_count)
# So we use the raw JSON with streaming, which extracts href_decoded
# The cache makes subsequent runs instant

cache_path_with_links = os.path.join(
    CONFIG.cache_dir, f"sentences_with_links_{CONFIG.max_articles}.pkl"
)

# Check if old cache exists WITHOUT links (from previous buggy run)
# If so, delete it to force re-extraction
if Path(cache_path_with_links).exists():
    print(f"📂 Found cache at {cache_path_with_links}")
    # Quick check: load and verify it has links
    try:
        with open(cache_path_with_links, "rb") as f:
            cached_data = pickle.load(f)
        has_links = any(s.links for s in cached_data[:100])
        if not has_links:
            print("   ⚠️  Cache has no link data! Deleting stale cache...")
            os.remove(cache_path_with_links)
            print("   🗑️  Stale cache deleted. Will re-extract with links.")
        else:
            print("   ✅ Cache has link data")
        del cached_data
        gc.collect()
    except Exception as e:
        print(f"   ⚠️  Error checking cache: {e}")
        print("   🗑️  Deleting potentially corrupted cache...")
        os.remove(cache_path_with_links)

# Always use the raw JSON loader to get actual links for Step 2
print("📌 Loading data WITH actual links (required for Step 2 evaluation)")
all_sentences = load_from_raw_json_streaming(
    CONFIG.raw_data_path,
    max_articles=CONFIG.max_articles,
    max_sentences=CONFIG.max_sentences,
    cache_path=cache_path_with_links,
)

print(f"\n📊 Dataset Statistics:")
print(f"   Total sentences: {len(all_sentences):,}")
link_counts = [s.link_count for s in all_sentences]
print(f"   Link count distribution:")
print(f"     Mean: {np.mean(link_counts):.2f}")
print(f"     Median: {np.median(link_counts):.2f}")
print(f"     Max: {np.max(link_counts)}")
print(
    f"     0 links: {sum(1 for c in link_counts if c == 0):,} ({100 * sum(1 for c in link_counts if c == 0) / len(link_counts):.1f}%)"
)
print(
    f"     1+ links: {sum(1 for c in link_counts if c >= 1):,} ({100 * sum(1 for c in link_counts if c >= 1) / len(link_counts):.1f}%)"
)

# Verify links are loaded (critical for Step 2!)
sentences_with_link_data = sum(1 for s in all_sentences if s.links)
total_link_objects = sum(len(s.links) for s in all_sentences)
print(f"\n🔗 Link Data Verification (for Step 2):")
print(f"   Sentences with link objects: {sentences_with_link_data:,}")
print(f"   Total link objects: {total_link_objects:,}")
if sentences_with_link_data == 0:
    print("   ⚠️  WARNING: No link data loaded! Step 2 evaluation will fail!")
    print("   ⚠️  Make sure to use raw JSON, not JSONL (which lacks link targets)")
else:
    # Show sample
    sample = next((s for s in all_sentences if s.links), None)
    if sample:
        print(f"   Sample link: {sample.links[0]}")

📌 Loading data WITH actual links (required for Step 2 evaluation)
📚 Reading JSONL file: ../articles_fr_withLinks.json


  0%|          | 0/2000 [00:00<?, ?it/s]

✅ Extracted 28,601 sentences with links

📊 Dataset Statistics:
   Total sentences: 28,601
   Link count distribution:
     Mean: 2.67
     Median: 2.00
     Max: 85
     0 links: 0 (0.0%)
     1+ links: 28,601 (100.0%)

🔗 Link Data Verification (for Step 2):
   Sentences with link objects: 28,601
   Total link objects: 76,435
   Sample link: {'anchor': 'Moulins', 'href_decoded': 'Moulins (Allier)', 'href_raw': 'Moulins%20%28Allier%29'}


In [23]:
# =============================================================================
# TRAIN / VALIDATION / TEST SPLIT
# =============================================================================


def split_data(
    data: List[SentenceData],
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> Tuple[List[SentenceData], List[SentenceData], List[SentenceData]]:
    """Split data into train/val/test sets."""

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    # Shuffle with seed
    random.seed(seed)
    shuffled = data.copy()
    random.shuffle(shuffled)

    n = len(shuffled)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    train_data = shuffled[:train_end]
    val_data = shuffled[train_end:val_end]
    test_data = shuffled[val_end:]

    return train_data, val_data, test_data


# Split the data
train_data, val_data, test_data = split_data(
    all_sentences,
    train_ratio=CONFIG.train_ratio,
    val_ratio=CONFIG.val_ratio,
    test_ratio=CONFIG.test_ratio,
    seed=CONFIG.seed,
)

print(f"\n📊 Data Split:")
print(
    f"   Train: {len(train_data):,} ({100 * len(train_data) / len(all_sentences):.1f}%)"
)
print(f"   Val:   {len(val_data):,} ({100 * len(val_data) / len(all_sentences):.1f}%)")
print(
    f"   Test:  {len(test_data):,} ({100 * len(test_data) / len(all_sentences):.1f}%)"
)


📊 Data Split:
   Train: 20,020 (70.0%)
   Val:   4,290 (15.0%)
   Test:  4,291 (15.0%)


---
# Part 5: Embedding Computation
---

In [24]:
# =============================================================================
# EMBEDDING MODEL
# =============================================================================

print(f"📦 Loading embedding model: {CONFIG.embedding_model}")
embedder = SentenceTransformer(CONFIG.embedding_model, device=CONFIG.device)
print(f"   Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

📦 Loading embedding model: intfloat/multilingual-e5-large
   Embedding dimension: 1024


In [25]:
# =============================================================================
# COMPUTE EMBEDDINGS
# =============================================================================


def compute_embeddings(
    data: List[SentenceData],
    embedder: SentenceTransformer,
    batch_size: int = 64,
    cache_path: Optional[str] = None,
) -> np.ndarray:
    """Compute embeddings for sentences."""

    # Check cache
    if cache_path and Path(cache_path).exists():
        print(f"📂 Loading cached embeddings from {cache_path}")
        return np.load(cache_path)

    print(f"🔢 Computing embeddings for {len(data):,} sentences...")

    sentences = [f"query: {s.sentence}" for s in data]  # E5 prefix

    embeddings = embedder.encode(
        sentences,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Cache if path provided
    if cache_path:
        print(f"💾 Caching embeddings to {cache_path}")
        np.save(cache_path, embeddings)

    return embeddings


# Compute embeddings for each split
train_emb_cache = os.path.join(
    CONFIG.cache_dir, f"train_embeddings_{CONFIG.max_articles}.npy"
)
val_emb_cache = os.path.join(
    CONFIG.cache_dir, f"val_embeddings_{CONFIG.max_articles}.npy"
)
test_emb_cache = os.path.join(
    CONFIG.cache_dir, f"test_embeddings_{CONFIG.max_articles}.npy"
)

train_embeddings = compute_embeddings(train_data, embedder, cache_path=train_emb_cache)
val_embeddings = compute_embeddings(val_data, embedder, cache_path=val_emb_cache)
test_embeddings = compute_embeddings(test_data, embedder, cache_path=test_emb_cache)

print(f"\n✅ Embeddings computed:")
print(f"   Train: {train_embeddings.shape}")
print(f"   Val:   {val_embeddings.shape}")
print(f"   Test:  {test_embeddings.shape}")

🔢 Computing embeddings for 20,020 sentences...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

💾 Caching embeddings to ./cache/train_embeddings_2000.npy
🔢 Computing embeddings for 4,290 sentences...


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

💾 Caching embeddings to ./cache/val_embeddings_2000.npy
🔢 Computing embeddings for 4,291 sentences...


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

💾 Caching embeddings to ./cache/test_embeddings_2000.npy

✅ Embeddings computed:
   Train: (20020, 1024)
   Val:   (4290, 1024)
   Test:  (4291, 1024)


---
# Part 6: Step 1 - Link Count Prediction Model
---

In [26]:
# =============================================================================
# MLP MODEL ARCHITECTURES
# =============================================================================


class LinkCountMLP(nn.Module):
    """
    Deep MLP for predicting Wikipedia link counts from sentence embeddings.

    Best architecture: 1024 -> 768 -> 512 -> 384 -> 256 -> 128 -> 1
    """

    def __init__(
        self,
        embed_dim: int = 1024,
        hidden_dims: List[int] = [1024, 768, 512, 384, 256, 128],
        dropout: float = 0.2,
    ):
        super().__init__()

        layers = []
        in_dim = embed_dim

        for h_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(in_dim, h_dim),
                    nn.LayerNorm(h_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            in_dim = h_dim

        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

        self.embed_dim = embed_dim
        self.hidden_dims = hidden_dims

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Predict with rounding to nearest integer."""
        self.eval()
        with torch.no_grad():
            pred = self.forward(x)
            return torch.clamp(pred.round(), min=0)


class ResidualMLP(nn.Module):
    """MLP with residual connections."""

    def __init__(
        self,
        embed_dim: int = 1024,
        hidden_dim: int = 512,
        n_blocks: int = 4,
        dropout: float = 0.15,
    ):
        super().__init__()

        self.input_proj = nn.Linear(embed_dim, hidden_dim)

        self.blocks = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.LayerNorm(hidden_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.LayerNorm(hidden_dim),
                )
                for _ in range(n_blocks)
            ]
        )

        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)

        for block in self.blocks:
            x = x + block(x)
            x = torch.nn.functional.gelu(x)

        return self.head(x).squeeze(-1)

In [27]:
# =============================================================================
# TRAINING FUNCTION
# =============================================================================


def train_link_count_model(
    train_embeddings: np.ndarray,
    train_labels: np.ndarray,
    val_embeddings: np.ndarray,
    val_labels: np.ndarray,
    model_class: type = LinkCountMLP,
    model_kwargs: Optional[Dict] = None,
    epochs: int = 150,
    batch_size: int = 256,
    learning_rate: float = 5e-4,
    weight_decay: float = 0.02,
    patience: int = 40,
    device: str = "cuda",
    save_path: Optional[str] = None,
) -> Tuple[nn.Module, Dict]:
    """
    Train a link count prediction model.

    Returns:
        Tuple of (trained model, training history)
    """
    print(f"\n{'=' * 60}")
    print("🚀 TRAINING LINK COUNT MODEL")
    print("=" * 60)

    # Create model
    model_kwargs = model_kwargs or {}
    model = model_class(**model_kwargs).to(device)

    params = sum(p.numel() for p in model.parameters())
    print(f"📐 Model: {model_class.__name__}")
    print(f"📐 Parameters: {params:,}")

    # Convert to tensors
    train_emb_t = torch.tensor(train_embeddings, dtype=torch.float32, device=device)
    train_lab_t = torch.tensor(train_labels, dtype=torch.float32, device=device)
    val_emb_t = torch.tensor(val_embeddings, dtype=torch.float32, device=device)
    val_lab_t = torch.tensor(val_labels, dtype=torch.float32, device=device)

    # Data loaders
    train_loader = DataLoader(
        TensorDataset(train_emb_t, train_lab_t),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(val_emb_t, val_lab_t),
        batch_size=batch_size * 2,
    )

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    # Training
    criterion = nn.L1Loss()  # MAE loss
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

    best_val_mae = float("inf")
    best_epoch = 0
    patience_counter = 0
    best_state_dict = None

    history: Dict[str, Any] = {"train_mae": [], "val_mae": [], "lr": []}

    start_time = time.time()

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0.0

        for data, target in train_loader:
            optimizer.zero_grad()

            if scaler:
                with torch.amp.autocast("cuda"):
                    output = model(data)
                    loss = criterion(output, target)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            train_loss += loss.item()

        scheduler.step()

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        lr = optimizer.param_groups[0]["lr"]

        history["train_mae"].append(avg_train)
        history["val_mae"].append(avg_val)
        history["lr"].append(lr)

        # Print progress
        if epoch % 10 == 0 or avg_val < best_val_mae:
            print(
                f"Epoch {epoch + 1:3d}/{epochs}: Train MAE={avg_train:.4f}, Val MAE={avg_val:.4f}, LR={lr:.2e}"
            )

        # Early stopping
        if avg_val < best_val_mae:
            best_val_mae = avg_val
            best_epoch = epoch + 1
            patience_counter = 0
            best_state_dict = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⏹️  Early stopping at epoch {epoch + 1}")
                break

    training_time = time.time() - start_time

    # Load best model
    if best_state_dict:
        model.load_state_dict(best_state_dict)

    # Save model
    if save_path:
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_class": model_class.__name__,
                "model_kwargs": model_kwargs,
                "best_epoch": best_epoch,
                "best_val_mae": best_val_mae,
            },
            save_path,
        )
        print(f"💾 Model saved to {save_path}")

    print(f"\n📊 Training Results:")
    print(f"   Best Epoch: {best_epoch}")
    print(f"   Best Val MAE: {best_val_mae:.4f}")
    print(f"   Training Time: {training_time / 60:.1f} minutes")

    history["best_epoch"] = best_epoch
    history["best_val_mae"] = best_val_mae
    history["training_time"] = training_time

    return model, history

In [28]:
# =============================================================================
# TRAIN THE MODEL
# =============================================================================

# Prepare labels
train_labels = np.array([s.link_count for s in train_data])
val_labels = np.array([s.link_count for s in val_data])
test_labels = np.array([s.link_count for s in test_data])

# Model save path
model_save_path = os.path.join(CONFIG.results_dir, "best_link_count_model.pt")

# Check if model exists
if Path(model_save_path).exists():
    print(f"📂 Loading existing model from {model_save_path}")
    checkpoint = torch.load(
        model_save_path, map_location=CONFIG.device, weights_only=False
    )

    link_count_model = LinkCountMLP(
        embed_dim=CONFIG.embed_dim,
        hidden_dims=[1024, 768, 512, 384, 256, 128],
        dropout=0.2,
    ).to(CONFIG.device)

    link_count_model.load_state_dict(checkpoint["model_state_dict"])
    print(f"   Best epoch: {checkpoint.get('best_epoch', 'N/A')}")
    print(f"   Best val MAE: {checkpoint.get('best_val_mae', 'N/A'):.4f}")
else:
    # Train new model
    link_count_model, training_history = train_link_count_model(
        train_embeddings=train_embeddings,
        train_labels=train_labels,
        val_embeddings=val_embeddings,
        val_labels=val_labels,
        model_class=LinkCountMLP,
        model_kwargs={
            "embed_dim": CONFIG.embed_dim,
            "hidden_dims": [1024, 768, 512, 384, 256, 128],
            "dropout": 0.2,
        },
        epochs=CONFIG.epochs,
        batch_size=CONFIG.batch_size,
        learning_rate=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
        patience=CONFIG.patience,
        device=CONFIG.device,
        save_path=model_save_path,
    )

📂 Loading existing model from ./results/best_link_count_model.pt
   Best epoch: 69
   Best val MAE: 0.7353


In [29]:
# =============================================================================
# EVALUATE STEP 1 ON ALL SPLITS
# =============================================================================


def evaluate_step1(
    model: nn.Module,
    embeddings: np.ndarray,
    labels: np.ndarray,
    device: str = "cuda",
    split_name: str = "Test",
) -> Step1Metrics:
    """Evaluate link count prediction model."""
    model.eval()

    emb_t = torch.tensor(embeddings, dtype=torch.float32, device=device)

    with torch.no_grad():
        predictions = model(emb_t).cpu().numpy()

    metrics = metrics_calc.calculate_step1_metrics(predictions, labels)
    metrics_calc.print_step1_metrics(metrics, title=f"STEP 1 - {split_name.upper()}")

    return metrics


# Evaluate on all splits
print("\n" + "=" * 70)
print("📊 STEP 1 EVALUATION: Link Count Prediction")
print("=" * 70)

step1_train_metrics = evaluate_step1(
    link_count_model, train_embeddings, train_labels, CONFIG.device, "Train"
)
step1_val_metrics = evaluate_step1(
    link_count_model, val_embeddings, val_labels, CONFIG.device, "Validation"
)
step1_test_metrics = evaluate_step1(
    link_count_model, test_embeddings, test_labels, CONFIG.device, "Test"
)


📊 STEP 1 EVALUATION: Link Count Prediction

📊 STEP 1 - TRAIN: Link Count Prediction
  Samples:            20,020
  MAE:                1.0880
  RMSE:               1.8277
  R-squared:          0.4122
  Correlation:        0.7031
------------------------------------------------------------
  Exact accuracy:     36.7%
  Within ±1:          76.3%
  Within ±2:          90.3%
  Within ±3:          95.7%

📊 STEP 1 - VALIDATION: Link Count Prediction
  Samples:            4,290
  MAE:                1.1133
  RMSE:               1.7848
  R-squared:          0.4540
  Correlation:        0.7347
------------------------------------------------------------
  Exact accuracy:     36.7%
  Within ±1:          75.3%
  Within ±2:          89.7%
  Within ±3:          95.4%

📊 STEP 1 - TEST: Link Count Prediction
  Samples:            4,291
  MAE:                1.0934
  RMSE:               1.6739
  R-squared:          0.4934
  Correlation:        0.7587
--------------------------------------------------

---
# Part 7: Step 2 - Link Retrieval with Qdrant
---

In [30]:
# =============================================================================
# QDRANT CONNECTION AND URL MAPPINGS
# =============================================================================

print(f"📦 Connecting to Qdrant at {CONFIG.qdrant_host}:{CONFIG.qdrant_port}")

try:
    qdrant_client = QdrantClient(
        host=CONFIG.qdrant_host, port=CONFIG.qdrant_port, prefer_grpc=True, timeout=1000
    )

    collection_info = qdrant_client.count(
        collection_name=CONFIG.collection_name, exact=True
    )
    print(
        f"✅ Connected! Collection '{CONFIG.collection_name}' has {collection_info.count:,} vectors"
    )
    QDRANT_AVAILABLE = True

except Exception as e:
    print(f"⚠️  Qdrant connection failed: {e}")
    print("   Step 2 evaluation will be skipped")
    QDRANT_AVAILABLE = False
    qdrant_client = None

📦 Connecting to Qdrant at localhost:6333
✅ Connected! Collection 'wikipedia_fr_chunks' has 2,800,041 vectors


In [31]:
# =============================================================================
# BUILD URL TO ID MAPPING
# =============================================================================


def build_url_to_id_mapping(
    client: QdrantClient, collection_name: str, cache_path: Optional[str] = None
) -> Tuple[Dict[str, int], Dict[int, str]]:
    """Build URL→ID and ID→Title mappings from Qdrant."""

    # Check cache
    if cache_path:
        url_cache = cache_path + "_url.pkl"
        title_cache = cache_path + "_title.pkl"

        if Path(url_cache).exists() and Path(title_cache).exists():
            print(f"📂 Loading cached mappings")
            with open(url_cache, "rb") as f:
                url_to_id = pickle.load(f)
            with open(title_cache, "rb") as f:
                id_to_title = pickle.load(f)
            print(f"   Loaded {len(url_to_id):,} URL mappings")
            return url_to_id, id_to_title

    print(f"🔨 Building URL→ID mapping from '{collection_name}'...")

    url_to_id = {}
    id_to_title = {}

    offset = None
    batch_size = 1000
    total_processed = 0
    seen_article_ids = set()

    while True:
        points, offset = client.scroll(
            collection_name=collection_name,
            limit=batch_size,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )

        if not points:
            break

        for point in points:
            article_id = point.payload.get("source_article_id")
            title = point.payload.get("source_article_title", "")

            if not article_id or not title or article_id in seen_article_ids:
                continue

            seen_article_ids.add(article_id)
            id_to_title[article_id] = title

            # Create URL variations
            patterns = [
                title,
                title.replace(" ", "_"),
                quote(title.replace(" ", "_"), safe=""),
                title.lower(),
                title.lower().replace(" ", "_"),
            ]

            for pattern in patterns:
                url_to_id[pattern] = article_id

        total_processed += len(points)

        if total_processed % 10000 == 0:
            print(f"   Processed {total_processed:,} chunks...")

        if offset is None:
            break

    print(
        f"✅ Created {len(url_to_id):,} URL mappings from {len(seen_article_ids):,} articles"
    )

    # Cache
    if cache_path:
        url_cache_path = cache_path + "_url.pkl"
        title_cache_path = cache_path + "_title.pkl"
        with open(url_cache_path, "wb") as f:
            pickle.dump(url_to_id, f)
        with open(title_cache_path, "wb") as f:
            pickle.dump(id_to_title, f)
        print(f"💾 Cached mappings")

    return url_to_id, id_to_title


# Build mappings
if QDRANT_AVAILABLE:
    mapping_cache = os.path.join(CONFIG.cache_dir, "url_mapping")
    url_to_id, id_to_title = build_url_to_id_mapping(
        qdrant_client, CONFIG.collection_name, cache_path=mapping_cache
    )
else:
    url_to_id = {}
    id_to_title = {}

📂 Loading cached mappings
   Loaded 9,196,030 URL mappings


In [32]:
# =============================================================================
# LINK RETRIEVAL FUNCTIONS
# =============================================================================


def resolve_url_to_id(href: str, url_to_id: Dict[str, int]) -> Optional[int]:
    """Resolve a URL/title to article ID."""
    if not href:
        return None

    # Try direct lookup
    target_id = url_to_id.get(href)
    if target_id:
        return target_id

    # Try variations
    for variation in [
        href.replace("_", " "),
        href.replace("%20", " "),
        href.lower(),
        href.lower().replace("_", " "),
    ]:
        target_id = url_to_id.get(variation)
        if target_id:
            return target_id

    return None


def retrieve_candidate_articles(
    client: QdrantClient,
    embeddings: np.ndarray,
    source_article_ids: List[int],
    collection_name: str,
    top_k: int = 20,
    min_similarity: float = 0.5,
) -> List[List[Dict]]:
    """
    Retrieve top-K candidate articles for each sentence.

    Returns list of candidate articles per sentence.
    """
    print(f"🔍 Retrieving top-{top_k} candidates for {len(embeddings):,} sentences...")

    all_candidates = []

    for i, (embedding, source_id) in enumerate(
        tqdm(
            zip(embeddings, source_article_ids),
            total=len(embeddings),
            desc="Retrieving",
        )
    ):
        # Search for similar chunks
        search_results = client.query_points(
            collection_name=collection_name,
            query=embedding.tolist(),
            limit=top_k * 3,  # Get more to aggregate by article
            score_threshold=min_similarity,
        ).points

        # Aggregate chunks by source article
        article_scores = {}
        for result in search_results:
            article_id = result.payload.get("source_article_id")
            article_title = result.payload.get(
                "source_article_title", f"ID {article_id}"
            )

            # Skip self-match
            if article_id == source_id:
                continue

            if article_id not in article_scores:
                article_scores[article_id] = {
                    "article_id": article_id,
                    "article_title": article_title,
                    "similarity_score": result.score,
                    "chunk_count": 1,
                }
            else:
                if result.score > article_scores[article_id]["similarity_score"]:
                    article_scores[article_id]["similarity_score"] = result.score
                article_scores[article_id]["chunk_count"] += 1

        # Sort by similarity and take top_k
        candidates = sorted(
            article_scores.values(), key=lambda x: x["similarity_score"], reverse=True
        )[:top_k]

        all_candidates.append(candidates)

    return all_candidates

In [33]:
# =============================================================================
# EVALUATE STEP 2: LINK RETRIEVAL
# =============================================================================


def evaluate_step2(
    client: QdrantClient,
    data: List[SentenceData],
    embeddings: np.ndarray,
    url_to_id: Dict[str, int],
    id_to_title: Dict[int, str],
    collection_name: str,
    top_k: int = 20,
    min_similarity: float = 0.5,
    split_name: str = "Test",
) -> Tuple[Step2Metrics, List[Dict]]:
    """Evaluate link retrieval on a dataset."""

    print(f"\n{'=' * 60}")
    print(f"📊 STEP 2 EVALUATION - {split_name.upper()}: Link Retrieval")
    print("=" * 60)

    # Get source article IDs
    source_ids = [s.article_id for s in data]

    # Retrieve candidates
    candidates = retrieve_candidate_articles(
        client,
        embeddings,
        source_ids,
        collection_name,
        top_k=top_k,
        min_similarity=min_similarity,
    )

    # Build ground truth IDs
    gt_ids_list = []
    for s in data:
        ids = []
        for link in s.links:
            href = link.get("href_decoded", "")
            target_id = resolve_url_to_id(href, url_to_id)
            if target_id and target_id != s.article_id:
                ids.append(target_id)
        gt_ids_list.append(ids)

    # Build prediction IDs
    pred_ids_list = [[c["article_id"] for c in cands] for cands in candidates]

    # Calculate metrics
    metrics = metrics_calc.calculate_step2_metrics(pred_ids_list, gt_ids_list)
    metrics_calc.print_step2_metrics(metrics, title=f"STEP 2 - {split_name.upper()}")

    # Build detailed results
    detailed_results = []
    for i, (s, cands, gt_ids, pred_ids) in enumerate(
        zip(data, candidates, gt_ids_list, pred_ids_list)
    ):
        gt_set = set(gt_ids)

        # Calculate per-sentence recall
        recall_at_k = {}
        for k in CONFIG.k_values:
            top_k_set = set(pred_ids[:k])
            hits = len(gt_set & top_k_set)
            recall_at_k[k] = hits / len(gt_set) if gt_set else 0

        detailed_results.append(
            {
                "sentence": s.sentence,
                "source_article": s.article_title,
                "ground_truth_count": len(gt_ids),
                "predicted_count": s.link_count,
                "ground_truth_ids": gt_ids,
                "predicted_ids": pred_ids,
                "candidates": cands,
                "recall_at_k": recall_at_k,
            }
        )

    return metrics, detailed_results


# Evaluate Step 2 on test set
if QDRANT_AVAILABLE:
    step2_test_metrics, step2_test_results = evaluate_step2(
        qdrant_client,
        test_data,
        test_embeddings,
        url_to_id,
        id_to_title,
        CONFIG.collection_name,
        top_k=CONFIG.top_k_retrieval,
        min_similarity=CONFIG.min_similarity,
        split_name="Test",
    )
else:
    print("⚠️  Skipping Step 2 evaluation (Qdrant not available)")
    step2_test_metrics = None
    step2_test_results = None


📊 STEP 2 EVALUATION - TEST: Link Retrieval
🔍 Retrieving top-20 candidates for 4,291 sentences...


Retrieving:   0%|          | 0/4291 [00:00<?, ?it/s]


📊 STEP 2 - TEST: Link Retrieval
  Sentences:              4,291
  Ground truth links:     9,608
------------------------------------------------------------
  Recall@1:               0.0821
  Recall@5:               0.1938
  Recall@10:              0.2580
  Recall@20:              0.3180
------------------------------------------------------------
  Precision@1:            0.1997
  Precision@5:            0.0943
  Precision@10:           0.0628
  Precision@20:           0.0387
------------------------------------------------------------
  MRR:                    0.1352
  Hit Rate@1:             18.4%
  Hit Rate@5:             35.3%
  Hit Rate@10:            43.0%
  Hit Rate@20:            49.3%


---
# Part 8: Detailed Analysis
---

In [34]:
# =============================================================================
# DISPLAY BEST AND WORST PREDICTIONS
# =============================================================================


def display_detailed_analysis(
    results: List[Dict],
    id_to_title: Dict[int, str],
    num_samples: int = 5,
    k_display: int = 5,
):
    """Show detailed examples of best and worst predictions."""

    if not results:
        print("No results to display")
        return

    # Sort by Recall@k_display
    sorted_results = sorted(
        results, key=lambda x: x["recall_at_k"].get(k_display, 0), reverse=True
    )

    print(f"\n{'=' * 80}")
    print(f"🎯 BEST PREDICTIONS (Highest Recall@{k_display})")
    print("=" * 80)

    for i, result in enumerate(sorted_results[:num_samples]):
        recall_k = result["recall_at_k"].get(k_display, 0)
        if recall_k == 0:
            continue

        print(f"\n{'─' * 80}")
        print(f"Example {i + 1}: Article '{result['source_article']}'")
        print(
            f"Sentence: '{result['sentence'][:100]}{'...' if len(result['sentence']) > 100 else ''}'"
        )
        print(
            f"Recall@1: {result['recall_at_k'].get(1, 0):.2f} | "
            f"Recall@5: {result['recall_at_k'].get(5, 0):.2f} | "
            f"Recall@10: {result['recall_at_k'].get(10, 0):.2f}"
        )

        gt_ids = set(result["ground_truth_ids"])
        pred_ids = result["predicted_ids"]

        print(f"\n✅ Ground Truth ({len(result['ground_truth_ids'])}):")
        for gt_id in result["ground_truth_ids"][:5]:
            title = id_to_title.get(gt_id, f"ID {gt_id}")
            rank = pred_ids.index(gt_id) + 1 if gt_id in pred_ids else None
            rank_str = f"(rank: {rank})" if rank else "(not found)"
            matched = "✓" if gt_id in set(pred_ids[:k_display]) else " "
            print(f"   [{matched}] {title} {rank_str}")

        print(f"\n🔮 Top {min(k_display, len(result['candidates']))} Predictions:")
        for j, cand in enumerate(result["candidates"][:k_display]):
            matched = "✓" if cand["article_id"] in gt_ids else "✗"
            print(
                f"   [{matched}] #{j + 1} {cand['article_title']} (sim: {cand['similarity_score']:.3f})"
            )

    print(f"\n{'=' * 80}")
    print(f"❌ WORST PREDICTIONS (Lowest Recall@{k_display} with predictions)")
    print("=" * 80)

    # Get worst predictions (with at least some predictions)
    worst = [r for r in sorted_results if len(r["candidates"]) > 0][-num_samples:]

    for i, result in enumerate(worst):
        print(f"\n{'─' * 80}")
        print(f"Example {i + 1}: Article '{result['source_article']}'")
        print(
            f"Sentence: '{result['sentence'][:100]}{'...' if len(result['sentence']) > 100 else ''}'"
        )
        print(
            f"Recall@1: {result['recall_at_k'].get(1, 0):.2f} | "
            f"Recall@5: {result['recall_at_k'].get(5, 0):.2f} | "
            f"Recall@10: {result['recall_at_k'].get(10, 0):.2f}"
        )

        gt_ids = set(result["ground_truth_ids"])
        pred_ids = result["predicted_ids"]

        print(f"\n✅ Ground Truth ({len(result['ground_truth_ids'])}):")
        for gt_id in result["ground_truth_ids"][:5]:
            title = id_to_title.get(gt_id, f"ID {gt_id}")
            rank = pred_ids.index(gt_id) + 1 if gt_id in pred_ids else None
            rank_str = f"(rank: {rank})" if rank else "(not found)"
            print(f"   {title} {rank_str}")

        print(f"\n🔮 Top {min(k_display, len(result['candidates']))} Predictions:")
        for j, cand in enumerate(result["candidates"][:k_display]):
            matched = "✓" if cand["article_id"] in gt_ids else "✗"
            print(
                f"   [{matched}] #{j + 1} {cand['article_title']} (sim: {cand['similarity_score']:.3f})"
            )


# Display analysis
if step2_test_results:
    display_detailed_analysis(
        step2_test_results, id_to_title, num_samples=5, k_display=5
    )


🎯 BEST PREDICTIONS (Highest Recall@5)

────────────────────────────────────────────────────────────────────────────────
Example 1: Article 'Iaidō'
Sentence: 'Autour de la pratique du sabre des samouraïs existaient deux types de "koryū" (écoles anciennes) com...'
Recall@1: 0.00 | Recall@5: 1.00 | Recall@10: 1.00

✅ Ground Truth (1):
   [✓] Kenjutsu (rank: 2)

🔮 Top 5 Predictions:
   [✗] #1 Aikiken (sim: 0.865)
   [✓] #2 Kenjutsu (sim: 0.863)
   [✗] #3 Liste des termes utilisés en aikido (sim: 0.858)
   [✗] #4 Samouraï (sim: 0.856)
   [✗] #5 Techniques de kenjutsu (sim: 0.855)

────────────────────────────────────────────────────────────────────────────────
Example 2: Article 'Oscar Wilde'
Sentence: 'Oscar Wilde naît au 21 Westland Row à Dublin (aujourd'hui le siège de l', Trinity College).'
Recall@1: 1.00 | Recall@5: 1.00 | Recall@10: 1.00

✅ Ground Truth (1):
   [✓] Dublin (rank: 1)

🔮 Top 5 Predictions:
   [✓] #1 Dublin (sim: 0.851)
   [✗] #2 Percival Wilde (sim: 0.846)
   [✗] #3 Wil

---
# Part 9: Summary and Final Report
---

In [35]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================


def print_summary_statistics(step2_metrics: Step2Metrics, step2_results: List[Dict]):
    """Print comprehensive summary statistics."""

    if not step2_results:
        print("No results to summarize")
        return

    # Hit rate analysis
    sentences_with_hit_at_1 = sum(
        1 for r in step2_results if r["recall_at_k"].get(1, 0) > 0
    )
    sentences_with_hit_at_5 = sum(
        1 for r in step2_results if r["recall_at_k"].get(5, 0) > 0
    )
    sentences_with_hit_at_10 = sum(
        1 for r in step2_results if r["recall_at_k"].get(10, 0) > 0
    )
    sentences_with_hit_at_20 = sum(
        1 for r in step2_results if r["recall_at_k"].get(20, 0) > 0
    )

    # Average recall per sentence
    avg_recall_at_1 = np.mean([r["recall_at_k"].get(1, 0) for r in step2_results])
    avg_recall_at_5 = np.mean([r["recall_at_k"].get(5, 0) for r in step2_results])
    avg_recall_at_10 = np.mean([r["recall_at_k"].get(10, 0) for r in step2_results])
    avg_recall_at_20 = np.mean([r["recall_at_k"].get(20, 0) for r in step2_results])

    n = len(step2_results)

    print(f"\n{'=' * 60}")
    print("📊 SUMMARY STATISTICS")
    print("=" * 60)

    print(f"\n🎯 Hit Rate (sentences with ≥1 correct in top-K):")
    print(
        f"   Hit@1:  {sentences_with_hit_at_1:,}/{n} ({100 * sentences_with_hit_at_1 / n:.1f}%)"
    )
    print(
        f"   Hit@5:  {sentences_with_hit_at_5:,}/{n} ({100 * sentences_with_hit_at_5 / n:.1f}%)"
    )
    print(
        f"   Hit@10: {sentences_with_hit_at_10:,}/{n} ({100 * sentences_with_hit_at_10 / n:.1f}%)"
    )
    print(
        f"   Hit@20: {sentences_with_hit_at_20:,}/{n} ({100 * sentences_with_hit_at_20 / n:.1f}%)"
    )

    print(f"\n📈 Average Per-Sentence Recall@K:")
    print(f"   Avg Recall@1:  {avg_recall_at_1:.4f}")
    print(f"   Avg Recall@5:  {avg_recall_at_5:.4f}")
    print(f"   Avg Recall@10: {avg_recall_at_10:.4f}")
    print(f"   Avg Recall@20: {avg_recall_at_20:.4f}")

    print(f"\n📊 Overall Metrics:")
    print(f"   MRR: {step2_metrics.mrr:.4f}")
    print(f"   Total ground truth links: {step2_metrics.n_ground_truth_links:,}")


if step2_test_metrics and step2_test_results:
    print_summary_statistics(step2_test_metrics, step2_test_results)


📊 SUMMARY STATISTICS

🎯 Hit Rate (sentences with ≥1 correct in top-K):
   Hit@1:  789/4291 (18.4%)
   Hit@5:  1,514/4291 (35.3%)
   Hit@10: 1,844/4291 (43.0%)
   Hit@20: 2,114/4291 (49.3%)

📈 Average Per-Sentence Recall@K:
   Avg Recall@1:  0.1020
   Avg Recall@5:  0.2193
   Avg Recall@10: 0.2801
   Avg Recall@20: 0.3334

📊 Overall Metrics:
   MRR: 0.1352
   Total ground truth links: 9,608


In [36]:
# =============================================================================
# FINAL REPORT
# =============================================================================


def generate_final_report(
    step1_train: Step1Metrics,
    step1_val: Step1Metrics,
    step1_test: Step1Metrics,
    step2_test: Optional[Step2Metrics],
    config: Config,
    save_path: Optional[str] = None,
) -> Dict:
    """Generate and save final report."""

    report = {
        "timestamp": datetime.now().isoformat(),
        "config": {
            "max_articles": config.max_articles,
            "train_ratio": config.train_ratio,
            "val_ratio": config.val_ratio,
            "test_ratio": config.test_ratio,
            "embedding_model": config.embedding_model,
            "top_k_retrieval": config.top_k_retrieval,
        },
        "step1_link_count_prediction": {
            "train": step1_train.to_dict(),
            "validation": step1_val.to_dict(),
            "test": step1_test.to_dict(),
        },
    }

    if step2_test:
        report["step2_link_retrieval"] = {
            "test": step2_test.to_dict(),
        }

    # Print summary
    print(f"\n{'=' * 70}")
    print("📋 FINAL REPORT")
    print("=" * 70)

    print(f"\n📊 STEP 1: Link Count Prediction")
    print(f"   {'Split':<12} {'MAE':<10} {'Exact Acc':<12} {'Within ±1':<12}")
    print(f"   {'-' * 46}")
    print(
        f"   {'Train':<12} {step1_train.mae:<10.4f} {step1_train.exact_accuracy * 100:<12.1f} {step1_train.within_1_accuracy * 100:<12.1f}"
    )
    print(
        f"   {'Validation':<12} {step1_val.mae:<10.4f} {step1_val.exact_accuracy * 100:<12.1f} {step1_val.within_1_accuracy * 100:<12.1f}"
    )
    print(
        f"   {'Test':<12} {step1_test.mae:<10.4f} {step1_test.exact_accuracy * 100:<12.1f} {step1_test.within_1_accuracy * 100:<12.1f}"
    )

    if step2_test:
        print(f"\n📊 STEP 2: Link Retrieval (Test Set)")
        print(f"   {'Metric':<15} {'Value':<10}")
        print(f"   {'-' * 25}")
        print(f"   {'Recall@1':<15} {step2_test.recall_at_1:<10.4f}")
        print(f"   {'Recall@5':<15} {step2_test.recall_at_5:<10.4f}")
        print(f"   {'Recall@10':<15} {step2_test.recall_at_10:<10.4f}")
        print(f"   {'Recall@20':<15} {step2_test.recall_at_20:<10.4f}")
        print(f"   {'MRR':<15} {step2_test.mrr:<10.4f}")
        print(f"   {'Hit Rate@10':<15} {step2_test.hit_rate_at_10 * 100:<10.1f}%")

    # Save report
    if save_path:
        with open(save_path, "w") as f:
            json.dump(report, f, indent=2)
        print(f"\n💾 Report saved to {save_path}")

    return report


# Generate final report
report_path = os.path.join(CONFIG.results_dir, "final_report.json")
final_report = generate_final_report(
    step1_train_metrics,
    step1_val_metrics,
    step1_test_metrics,
    step2_test_metrics,
    CONFIG,
    save_path=report_path,
)


📋 FINAL REPORT

📊 STEP 1: Link Count Prediction
   Split        MAE        Exact Acc    Within ±1   
   ----------------------------------------------
   Train        1.0880     36.7         76.3        
   Validation   1.1133     36.7         75.3        
   Test         1.0934     36.3         75.4        

📊 STEP 2: Link Retrieval (Test Set)
   Metric          Value     
   -------------------------
   Recall@1        0.0821    
   Recall@5        0.1938    
   Recall@10       0.2580    
   Recall@20       0.3180    
   MRR             0.1352    
   Hit Rate@10     43.0      %

💾 Report saved to ./results/final_report.json


---
# Part 10: Demo - Process New Article
---

In [37]:
# =============================================================================
# DEMO: PROCESS A NEW ARTICLE
# =============================================================================


def process_article(
    text: str,
    embedder: SentenceTransformer,
    link_count_model: nn.Module,
    qdrant_client: Optional[QdrantClient],
    url_to_id: Dict[str, int],
    id_to_title: Dict[int, str],
    collection_name: str,
    top_k: int = 10,
    device: str = "cuda",
) -> Dict:
    """
    Process a Wikipedia article and predict links.

    Returns dict with sentences, predicted link counts, and candidate articles.
    """
    # Clean and split text
    clean_text, _ = text_processor.extract_links_and_clean_text(text)
    sentences = text_processor.split_into_sentences(clean_text)

    if not sentences:
        return {"sentences": [], "total_predicted_links": 0}

    # Get embeddings
    prefixed = [f"query: {s}" for s in sentences]
    embeddings = embedder.encode(
        prefixed, convert_to_numpy=True, normalize_embeddings=True
    )

    # Predict link counts
    link_count_model.eval()
    emb_t = torch.tensor(embeddings, dtype=torch.float32, device=device)
    with torch.no_grad():
        pred_counts = (
            torch.clamp(link_count_model(emb_t).round(), min=0).int().cpu().numpy()
        )

    # Retrieve candidates if Qdrant available
    candidates_list = []
    if qdrant_client:
        for embedding in embeddings:
            search_results = qdrant_client.query_points(
                collection_name=collection_name,
                query=embedding.tolist(),
                limit=top_k * 2,
                score_threshold=0.5,
            ).points

            article_scores = {}
            for result in search_results:
                article_id = result.payload.get("source_article_id")
                article_title = result.payload.get(
                    "source_article_title", f"ID {article_id}"
                )

                if article_id not in article_scores:
                    article_scores[article_id] = {
                        "article_id": article_id,
                        "article_title": article_title,
                        "similarity_score": result.score,
                    }
                elif result.score > article_scores[article_id]["similarity_score"]:
                    article_scores[article_id]["similarity_score"] = result.score

            candidates = sorted(
                article_scores.values(),
                key=lambda x: x["similarity_score"],
                reverse=True,
            )[:top_k]
            candidates_list.append(candidates)

    # Build results
    results = []
    for i, (sent, count) in enumerate(zip(sentences, pred_counts)):
        result = {
            "sentence": sent,
            "predicted_link_count": int(count),
            "candidates": candidates_list[i] if candidates_list else [],
        }
        results.append(result)

    return {
        "sentences": results,
        "total_predicted_links": int(pred_counts.sum()),
    }


# Demo with example text
example_text = """
Paris est la capitale de la France et la ville la plus peuplée du pays.
Elle est située au coeur de la région Île-de-France, sur la Seine.
La Tour Eiffel est le monument le plus célèbre de Paris.
Elle a été construite par Gustave Eiffel pour l'Exposition universelle de 1889.
Le Louvre est le plus grand musée d'art au monde.
Il abrite des oeuvres célèbres comme la Joconde de Léonard de Vinci.
"""

print(f"\n{'=' * 70}")
print("🔗 DEMO: Processing Example Article")
print("=" * 70)
print(f"\n📄 Input text:\n{example_text.strip()}")

demo_result = process_article(
    example_text,
    embedder,
    link_count_model,
    qdrant_client if QDRANT_AVAILABLE else None,
    url_to_id,
    id_to_title,
    CONFIG.collection_name,
    top_k=5,
    device=CONFIG.device,
)

print(f"\n📊 Predictions:")
print("-" * 70)
for i, sent_result in enumerate(demo_result["sentences"]):
    count = sent_result["predicted_link_count"]
    sentence = (
        sent_result["sentence"][:60] + "..."
        if len(sent_result["sentence"]) > 60
        else sent_result["sentence"]
    )
    print(f"\n  [{count} links] {sentence}")

    if sent_result["candidates"]:
        print(f"    Top candidates:")
        for cand in sent_result["candidates"][:3]:
            print(
                f"      → {cand['article_title']} (sim: {cand['similarity_score']:.3f})"
            )

print(f"\n📈 Total predicted links: {demo_result['total_predicted_links']}")


🔗 DEMO: Processing Example Article

📄 Input text:
Paris est la capitale de la France et la ville la plus peuplée du pays.
Elle est située au coeur de la région Île-de-France, sur la Seine.
La Tour Eiffel est le monument le plus célèbre de Paris.
Elle a été construite par Gustave Eiffel pour l'Exposition universelle de 1889.
Le Louvre est le plus grand musée d'art au monde.
Il abrite des oeuvres célèbres comme la Joconde de Léonard de Vinci.

📊 Predictions:
----------------------------------------------------------------------

  [3 links] Paris est la capitale de la France et la ville la plus peupl...
    Top candidates:
      → Paris (Texas) (sim: 0.858)
      → Paris (Illinois) (sim: 0.857)
      → Paris (Arkansas) (sim: 0.856)

  [2 links] Elle est située au coeur de la région Île-de-France, sur la ...
    Top candidates:
      → Liste des sites classés et inscrits de la Seine-Saint-Denis (sim: 0.852)
      → Conseil général de la Seine (sim: 0.844)
      → Tour Rive Gauche (sim: 0

---
# Part 11: Retraining (Optional)
---

In [38]:
# =============================================================================
# RETRAIN MODEL (Optional - set RETRAIN = True to run)
# =============================================================================

RETRAIN = False  # Set to True to retrain the model

if RETRAIN:
    print("\n🔄 RETRAINING MODEL...")

    # Delete existing model
    if Path(model_save_path).exists():
        os.remove(model_save_path)

    # Train new model
    link_count_model, training_history = train_link_count_model(
        train_embeddings=train_embeddings,
        train_labels=train_labels,
        val_embeddings=val_embeddings,
        val_labels=val_labels,
        model_class=LinkCountMLP,
        model_kwargs={
            "embed_dim": CONFIG.embed_dim,
            "hidden_dims": [1024, 768, 512, 384, 256, 128],
            "dropout": 0.2,
        },
        epochs=CONFIG.epochs,
        batch_size=CONFIG.batch_size,
        learning_rate=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
        patience=CONFIG.patience,
        device=CONFIG.device,
        save_path=model_save_path,
    )

    # Re-evaluate
    print("\n📊 Re-evaluating after retraining...")
    step1_test_metrics = evaluate_step1(
        link_count_model, test_embeddings, test_labels, CONFIG.device, "Test"
    )

---
# End of Notebook
---

## Summary

This notebook implements a complete Wikipedia link prediction pipeline:

1. **Step 1 - Link Count Prediction**: MLP model predicts how many links each sentence should contain
   - Metrics: MAE, RMSE, Exact Accuracy, Within ±1/2/3 Accuracy

2. **Step 2 - Link Retrieval**: Qdrant vector search retrieves top-K candidate articles
   - Metrics: Recall@K, Precision@K, MRR, Hit Rate

## Files Generated
- `cache/`: Cached embeddings and mappings
- `results/best_link_count_model.pt`: Trained MLP model
- `results/final_report.json`: Evaluation results

## To Retrain
Set `RETRAIN = True` in Part 11 and run that cell.

In [39]:
print("\n✅ Notebook execution complete!")
print(f"📁 Results saved to: {CONFIG.results_dir}")


✅ Notebook execution complete!
📁 Results saved to: ./results
